# White rust pixel map – XGBoost, task C: severity score (0–9)
UAV hyperspectral image of 24 February 2026 · model trained on the 2025-26 ASD leaf spectra (leave-one-replication-out evaluation)

| Item | This notebook |
|---|---|
| Model | **XGBoost** (machine learning), file `C_XGBoost.npz` |
| Why this model | best overall and best ML on the leaves |
| Input given to the model | direct: direct UAV reflectance |
| Features (computed from each pixel spectrum) | FD1090, FD1060, FD870, FD595, MCARI, Dr |
| Accuracy on the 48 UAV plot spectra | R² -0.113 |
| Output rasters | `C_XGBoost` (predicted severity score 0–9) – ENVI `.dat/.hdr`, GeoTIFF (if `rasterio` is installed) and a PNG map |

On the 48 UAV plots this model was below chance, so the map is **not validated** and must not be read as severity – it shows what the leaf model predicts.

**Steps.** 0 install packages · 1 settings · 2 libraries and image helpers · 3 reference data · features · load the model · self-check on the 48 plot means · open the image and crop mask · pixel loop · map · per-plot accuracy · outputs.

**Requirements.** Python 3.9+, `numpy`, `scipy`, `pandas`, `matplotlib`; optional `rasterio` for GeoTIFF. The model is stored as plain arrays and predicted with NumPy exactly as scikit-learn/xgboost do, so their versions do not matter.
**Time.** Run first with `SPATIAL_STEP = 4` (preview, about 1/16 of the pixels), then `SPATIAL_STEP = 1` for the final map.

## 0 Install the packages (run once)
Run this cell once. If it installs something new, restart the kernel (menu *Kernel → Restart*) and continue with step 1. Later runs can skip it.

In [ ]:
%pip install -q numpy scipy pandas matplotlib rasterio

## 1 Settings
Your paths are already filled in. The results go to a sub-folder of `OUT_DIR` named after the model. If the model file is not found at `MODEL_FILE`, the notebook also looks in a `models` folder next to the notebook.

In [ ]:
# ============================ SETTINGS ============================
HDR_FILE  = r'D:\Coalighned\24_02_2026\subset_200526.hdr'          # ENVI header of the 24 Feb image (the data file 'subset_200526', ~9 GB, must be next to it)
DATA_NAME = 'subset_200526'                                         # name of the image data file
ROI_FILE  = r'D:\Coalighned\24_02_2026\48_plot_roi_new.xml'        # 48 plot ROIs, ROI #n = plot n (None = skip the per-plot accuracy)
OUT_DIR   = r'C:\Users\BDI LAB\Desktop\ASD_MODEL_ON_240226_UAV_IMAGE_PIXELWISE\IMAGE_VALIDATED'   # your output folder
MODEL_FILE = r'C:\Users\BDI LAB\Desktop\ASD_MODEL_ON_240226_UAV_IMAGE_PIXELWISE\models\C_XGBoost.npz'   # this model (from the kit's 'models' folder)
PLOT_AVG_FILE = r'D:\Coalighned\24_02_2026\envi_plot_avg.txt'      # ENVI plot-average file for the self-check (skipped if the file is not there)
OUT_NAME  = 'C_XGBoost'                                                 # sub-folder of OUT_DIR for this model's results

SPATIAL_STEP = 4        # 4 = quick preview; 1 = final full-resolution map
BLOCK_ROWS   = 16       # image rows processed at a time (reduce if memory is short)
NDVI_MIN     = 0.6      # crop mask: NDVI(898, 684) > 0.6 as in the methodology
REFLECTANCE_SCALE = 'auto'  # 'auto' (divides by 10000 if the data look like 0-10000) or a number

## 2 Libraries and image helpers
The image is memory-mapped: only the rows being processed are read, so a 9 GB image runs in a few GB of memory. The ROI parser converts the plot polygons from map coordinates to image pixels.

In [ ]:
import os, re, json, time, math, warnings
import xml.etree.ElementTree as ET
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.path import Path as MplPath
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.signal import savgol_coeffs
warnings.filterwarnings('ignore', category=RuntimeWarning)
try:
    from IPython.display import Image as _Img, display as _display
    def show(path, width=900): _display(_Img(path, width=width))
except Exception:
    def show(path, width=900): print('saved', path)

assert os.path.exists(HDR_FILE), f'Header not found: {HDR_FILE}'
SRC = os.path.dirname(HDR_FILE); _base = os.path.splitext(HDR_FILE)[0]
DATA_FILE = next((c for c in [os.path.join(SRC, DATA_NAME), _base, _base + '.dat', _base + '.img', _base + '.bil', _base + '.raw']
                  if os.path.isfile(c) and not c.lower().endswith('.hdr')), None)
if DATA_FILE is None:
    raise FileNotFoundError(f'Image data file not found next to {HDR_FILE}. Files there: {os.listdir(SRC)}')
OUT = os.path.join(OUT_DIR, OUT_NAME); os.makedirs(OUT, exist_ok=True)
_mf = os.path.basename(MODEL_FILE.replace('\\', '/'))
_cands = [MODEL_FILE, os.path.join(os.getcwd(), 'models', _mf), os.path.join(os.getcwd(), _mf), os.path.join(os.path.dirname(OUT_DIR), 'models', _mf)]
MODEL_FILE = next((c for c in _cands if os.path.isfile(c)), None)
if MODEL_FILE is None: raise FileNotFoundError('Model file ' + _mf + ' not found. Looked in:\n  ' + '\n  '.join(_cands) + '\nCopy the kit folder "models" to one of these places or correct MODEL_FILE in step 1.')
print('model file  :', MODEL_FILE); print('results to  :', OUT)

ENVI_DTYPES = {1: np.uint8, 2: np.int16, 3: np.int32, 4: np.float32, 5: np.float64, 12: np.uint16, 13: np.uint32, 14: np.int64, 15: np.uint64}
def read_envi_header(path):
    txt = open(path, encoding='latin-1').read(); hdr = {}
    for m in re.finditer(r'^\s*([^=\n{}]+?)\s*=\s*(\{[^}]*\}|[^\n]*)', txt, re.M):
        v = m.group(2).strip(); hdr[m.group(1).strip().lower()] = v[1:-1].strip() if v.startswith('{') else v
    return hdr
class EnviCube:
    # memory-mapped input image; reads only what is needed
    def __init__(self, path, hdr):
        self.ns, self.nl, self.nb = int(hdr['samples']), int(hdr['lines']), int(hdr['bands'])
        dt = np.dtype(ENVI_DTYPES[int(hdr['data type'])])
        if int(hdr.get('byte order', '0')) == 1: dt = dt.newbyteorder('>')
        self.il = hdr.get('interleave', 'bsq').strip().lower()
        shape = {'bil': (self.nl, self.nb, self.ns), 'bsq': (self.nb, self.nl, self.ns), 'bip': (self.nl, self.ns, self.nb)}[self.il]
        off = int(hdr.get('header offset', '0'))
        if os.path.getsize(path) < off + int(np.prod(shape)) * dt.itemsize: raise ValueError('data file is smaller than the header says')
        self.mm = np.memmap(path, dtype=dt, mode='r', offset=off, shape=shape)
    def rows(self, r0, r1, step):
        # rows r0..r1 (full-res, every step-th row and column) -> (rows, cols, bands) float32
        ys, xs = slice(r0, r1, step), slice(0, self.ns, step)
        if self.il == 'bil': a = np.asarray(self.mm[ys, :, xs]).transpose(0, 2, 1)
        elif self.il == 'bsq': a = np.asarray(self.mm[:, ys, xs]).transpose(1, 2, 0)
        else: a = np.asarray(self.mm[ys, xs])
        return a.astype(np.float32)
class Geo:
    def __init__(self, hdr):
        mi = [s.strip() for s in hdr['map info'].split(',')]; self.items, self.proj = mi, mi[0]
        self.refx, self.refy, self.e0, self.n0, self.dx, self.dy = map(float, mi[1:7]); self.csys = hdr.get('coordinate system string', '').strip()
        g = self.proj.lower().startswith('geographic')
        self.px_w = self.dx * (111320.0 * np.cos(np.radians(self.n0)) if g else 1.0); self.px_h = self.dy * (110574.0 if g else 1.0)
    def to_file(self, x, y):
        x, y = np.asarray(x, float), np.asarray(y, float); return (x - self.e0) / self.dx + (self.refx - 1), (self.n0 - y) / self.dy + (self.refy - 1)
    def to_map(self, fx, fy):
        fx, fy = np.asarray(fx, float), np.asarray(fy, float); return self.e0 + (fx - (self.refx - 1)) * self.dx, self.n0 - (fy - (self.refy - 1)) * self.dy
    def map_info(self, step=1):
        e, n = self.to_map(0.5 - 0.5 * step, 0.5 - 0.5 * step)          # output pixel centres = sampled input pixel centres
        return '{' + ', '.join([self.proj, '1.0000', '1.0000', f'{float(e):.10f}', f'{float(n):.10f}', repr(self.dx * step), repr(self.dy * step)] + self.items[7:]) + '}'
def write_envi(path, arr2d, band_name, dtype_code, extra=None):
    arr2d[:, None, :].astype({4: '<f4', 1: 'u1'}[dtype_code]).tofile(path)
    L = ['ENVI', f'description = {{{os.path.basename(path)}}}', f'samples = {arr2d.shape[1]}', f'lines   = {arr2d.shape[0]}', 'bands   = 1', 'header offset = 0',
         'file type = ENVI Standard', f'data type = {dtype_code}', 'interleave = bil', 'byte order = 0', f'map info = {geo.map_info(SPATIAL_STEP)}', 'band names = {' + band_name + '}']
    if geo.csys: L.append('coordinate system string = {' + geo.csys + '}')
    for k, v in (extra or {}).items(): L.append(f'{k} = {v}')
    open(os.path.splitext(path)[0] + '.hdr', 'w').write('\n'.join(L) + '\n')
def write_tif(path, arr2d, nodata):
    try:
        import rasterio
        from rasterio.transform import from_origin
    except ImportError:
        return False
    e0, n0 = geo.to_map(0.5 - 0.5 * SPATIAL_STEP, 0.5 - 0.5 * SPATIAL_STEP)
    with rasterio.open(path, 'w', driver='GTiff', height=arr2d.shape[0], width=arr2d.shape[1], count=1, dtype=str(arr2d.dtype), crs='EPSG:4326',
                       transform=from_origin(float(e0), float(n0), geo.dx * SPATIAL_STEP, geo.dy * SPATIAL_STEP), nodata=nodata) as dst:
        dst.write(arr2d, 1)
    return True
def read_roi_xml(path):
    # ENVI ROI .xml -> {plot number: [polygons in full-res file coordinates]}
    plots = {}
    for i, reg in enumerate(ET.parse(path).getroot().iter('Region')):
        m = re.search(r'(\d+)\D*$', reg.get('name', '')); pno = int(m.group(1)) if m else i + 1; polys = []
        for gd in reg.iter('GeometryDef'):
            cs = (gd.findtext('CoordSysStr') or 'none').strip()
            for poly in gd.iter('Polygon'):
                c = poly.find('.//Exterior//Coordinates')
                if c is None or not (c.text or '').strip(): continue
                v = np.array(c.text.split(), float).reshape(-1, 2)
                fx, fy = (v[:, 0], v[:, 1]) if cs.lower() == 'none' else geo.to_file(v[:, 0], v[:, 1])
                polys.append(np.column_stack([fx, fy]))
        if polys: plots[pno] = polys
    return dict(sorted(plots.items()))
hdr = read_envi_header(HDR_FILE); cube = EnviCube(DATA_FILE, hdr); geo = Geo(hdr)
WL = np.array([float(t) for t in hdr['wavelength'].split(',')])
print(f'image {cube.ns} x {cube.nl} px, {cube.nb} bands ({cube.il}), pixel {geo.px_w * 100:.2f} x {geo.px_h * 100:.2f} cm, {WL[0]:.1f}-{WL[-1]:.1f} nm')

## 3 Reference data (embedded)
`TRUTH` = ASD leaf scores of 24 Feb 2026 for the 48 plots; `EXPECTED` = this model's prediction for the 48 plot-mean spectra (self-check). This model uses the UAV reflectance **directly**, so no reference spectra are needed.

In [ ]:
# ASD leaf scores of 24 Feb 2026 for the 48 plots (plot = ROI number)
TRUTH = pd.DataFrame({'plot': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48],
    'block': ["C", "C", "C", "C", "C", "C", "C", "C", "C", "C", "C", "C", "R3", "R3", "R3", "R3", "R3", "R3", "R3", "R3", "R3", "R3", "R3", "R3", "R2", "R2", "R2", "R2", "R2", "R2", "R2", "R2", "R2", "R2", "R2", "R2", "R1", "R1", "R1", "R1", "R1", "R1", "R1", "R1", "R1", "R1", "R1", "R1"],
    'status': ["Healthy", "Healthy", "Healthy", "Healthy", "Healthy", "Healthy", "Healthy", "Healthy", "Healthy", "Healthy", "Healthy", "Healthy", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased", "Diseased"],
    'variety': ["V2", "V2", "V3", "V1", "V2", "V3", "V1", "V3", "V1", "V3", "V2", "V1", "V2", "V1", "V1", "V3", "V2", "V2", "V2", "V3", "V3", "V1", "V3", "V1", "V3", "V2", "V3", "V1", "V3", "V2", "V1", "V2", "V3", "V1", "V2", "V1", "V1", "V3", "V2", "V3", "V1", "V2", "V1", "V2", "V2", "V3", "V1", "V3"],
    'sowing': ["D4", "D2", "D1", "D3", "D1", "D4", "D1", "D2", "D2", "D3", "D3", "D4", "D1", "D4", "D3", "D2", "D3", "D4", "D2", "D4", "D3", "D2", "D1", "D1", "D1", "D4", "D2", "D4", "D3", "D1", "D2", "D3", "D4", "D1", "D2", "D3", "D3", "D1", "D2", "D2", "D1", "D1", "D2", "D3", "D4", "D4", "D4", "D3"],
    'mean_score': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.0, 3.4, 2.6, 2.8667, 2.8, 2.4, 4.2, 4.0, 2.6, 3.4, 3.1333, 6.0667, 4.0667, 3.8, 3.5333, 4.0, 4.2, 5.8, 5.6, 4.0, 3.6, 3.9333, 5.2667, 5.2, 4.8, 3.9333, 4.6, 4.6, 5.2667, 5.9333, 3.8, 3.8, 2.4667, 3.2, 5.2, 3.8],
    'grade': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 5, 3, 3, 3, 3, 7, 5, 3, 3, 3, 5, 5, 5, 3, 3, 3, 5, 5, 5, 3, 5, 5, 5, 5, 3, 3, 3, 3, 5, 3],
    'PDI': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 33.333, 37.778, 28.889, 31.852, 31.111, 26.667, 46.667, 44.444, 28.889, 37.778, 34.815, 67.407, 45.185, 42.222, 39.259, 44.444, 46.667, 64.444, 62.222, 44.444, 40.0, 43.704, 58.519, 57.778, 53.333, 43.704, 51.111, 51.111, 58.519, 65.926, 42.222, 42.222, 27.407, 35.556, 57.778, 42.222]})
# this model's prediction for the 48 plot-mean spectra (used by the self-check)
EXPECTED = [2.209183, 2.514191, 2.286079, 1.854475, 2.790873, 2.524873, 2.339098, 3.560809, 2.896771, 1.827912, 2.870884, 2.809797, 1.974827, 1.630756, 2.383191, 2.405436, 1.815733, 2.588126, 2.509863, 2.755686, 2.16085, 2.339098, 2.943101, 2.943101, 2.646797, 1.953017, 2.263916, 2.391958, 3.583213, 2.126366, 2.265552, 4.193283, 3.0833, 2.943101, 2.094151, 2.772026, 2.082396, 2.943101, 2.701328, 2.790873, 2.484134, 2.484134, 2.250667, 2.281436, 2.917932, 2.365741, 2.129346, 3.643649]
MODEL = dict(task='C', model='XGBoost', variant='direct', uav_metric='R2', uav_value=-0.1127)
GRADES = [0, 1, 3, 5, 7, 9]; KEY, TASK, VARIANT = 'C_XGBoost', MODEL['task'], MODEL['variant']
assert len(WL) == 483 and abs(WL[0] - 399.38) < 0.5 and abs(WL[-1] - 2397.55) < 0.5, 'the image bands differ from the 24 Feb image the model inputs were built for'
print(f"{MODEL['model']} – task {TASK}, input '{VARIANT}' (reflectance used as it is), accuracy on the 48 UAV plots: {MODEL['uav_metric']} {MODEL['uav_value']:.3f}")

## 4 Features – identical to the ASD pipeline
For every pixel: (1) the input variant; (2) resampling to 1 nm, 350–2500 nm (gaps bridged linearly); (3) Savitzky–Golay smoothing / first derivative (window 11, order 2; Savitzky and Golay, 1964); (4) the model's features: first derivatives FD at given wavelengths, red-edge position REP and amplitude Dr (680–760 nm; Horler et al., 1983; Filella and Peñuelas, 1994), DWSI-2 = R1660/R550 (Apan et al., 2004), MCARI (Daughtry et al., 2000). Steps 2–3 are linear and are combined into one matrix evaluated only at the needed wavelengths – the same values as the ASD code, fast and light on memory.

In [ ]:
FULL = np.arange(350, 2501).astype(float)
_I = np.eye(len(WL)); MI = np.stack([np.interp(FULL, WL, _I[j]) for j in range(len(WL))], 1)   # (2151, bands): identical to np.interp per pixel
C0, C1 = savgol_coeffs(11, 2, deriv=0, use='dot'), savgol_coeffs(11, 2, deriv=1, use='dot')
def _op(w, c):
    v = np.zeros(len(FULL)); i = int(round(w - 350)); v[i - 5:i + 6] = c; return v
def build_operator(names):
    cols, keys = [], []
    def need(key, w, c):
        if key not in keys: cols.append(_op(w, c)); keys.append(key)
    for n in names:
        if n.startswith('FD') and n[2:].isdigit(): need(n, int(n[2:]), C1)
        elif n.startswith('R') and n[1:].isdigit(): need(n, int(n[1:]), C0)
        elif n in ('REP', 'Dr'):
            for w in range(680, 761): need('FD%d' % w, w, C1)
        elif n == 'DWSI-2': need('R1660', 1660, C0); need('R550', 550, C0)
        elif n == 'MCARI': need('R700', 700, C0); need('R670', 670, C0); need('R550', 550, C0)
        else: raise ValueError('feature not implemented: ' + n)
    return MI.T @ np.column_stack(cols), keys
def make_variant(S, v):
    return S                                   # this model uses the reflectance directly
def features(S, names, OP):
    M, keys = OP; Y = S.astype(np.float64) @ M; c = {k: j for j, k in enumerate(keys)}; g = lambda k: Y[:, c[k]]; out = []
    for n in names:
        if n in ('REP', 'Dr'):
            red = Y[:, [c['FD%d' % w] for w in range(680, 761)]]
            out.append(680 + red.argmax(1).astype(float) if n == 'REP' else red.max(1) * 1000)
        elif n == 'DWSI-2': out.append(g('R1660') / g('R550'))
        elif n == 'MCARI': out.append(((g('R700') - g('R670')) - 0.2 * (g('R700') - g('R550'))) * (g('R700') / g('R670')))
        else: out.append(g(n))
    return np.column_stack(out)

## 5 Load the model
Arrays from the `.npz` file; the prediction below reproduces scikit-learn/xgboost exactly. Inputs are standardised with the training mean and SD stored in the file.

In [ ]:
z = np.load(MODEL_FILE, allow_pickle=False); FEATURES = [str(f) for f in z['features']]; MU, SD = z['mu'], z['sd']
FEAT, THR, YES, NO, LEAF, ISLEAF, OFFSET, BASE = z['feat'], z['thr'], z['yes'], z['no'], z['leaf'], z['isleaf'], z['offset'], float(z['base'][0])
def predict(S):
    # spectra -> severity score: base score + sum of the leaf values of all boosted trees (XGBoost), clipped to 0-9
    Z = ((features(make_variant(S, VARIANT), FEATURES, OP) - MU) / SD).astype(np.float32); n = len(Z); rows = np.arange(n); v = np.full(n, BASE)
    for o in OFFSET:
        node = np.zeros(n, int)
        while True:
            g = o + node; lf = ISLEAF[g]
            if lf.all(): break
            go = Z[rows, FEAT[g]] < THR[g]; node = np.where(lf, node, np.where(go, YES[g], NO[g]))
        v += LEAF[o + node]
    v = np.clip(v, 0, 9); return v, v
print(f'XGBoost: {len(OFFSET)} trees, base score {BASE:.4f}; features: {", ".join(FEATURES)}')
OP = build_operator(FEATURES)

## 6 Self-check on the 48 plot-mean spectra
The model predicts the 48 plot means of the ENVI plot-average file; the result must agree with `EXPECTED` (computed when this notebook was built). A maximum difference below 0.001 means your computer reproduces the analysis exactly.

In [ ]:
if PLOT_AVG_FILE and os.path.exists(PLOT_AVG_FILE):
    L = open(PLOT_AVG_FILE).read().splitlines(); nh = sum(l.startswith('Column') for l in L)
    A = np.array([[float(x) for x in l.split()] for l in L[nh + 1:] if l.strip()])
    assert np.abs(A[:, 0] - WL).max() < 0.01, 'plot-average wavelengths differ from the image'
    d = np.abs(predict(A[:, 1:].T.astype(np.float32))[0] - np.array(EXPECTED)).max()
    print(f'self-check: max difference {d:.2e}', '-> OK' if d < 1e-3 else '-> CHECK library versions')
else:
    print('self-check skipped (no plot-average file)')

## 7 Open the image and build the crop mask
Pixels outside the field are 0 in the masked image. A pixel is used when its reflectance is valid and NDVI = (R898 − R684)/(R898 + R684) > `NDVI_MIN`. Reflectance is rescaled automatically if stored as 0–10000.

In [ ]:
ST = SPATIAL_STEP; NS, NL = len(range(0, cube.ns, ST)), len(range(0, cube.nl, ST))
bi = lambda w: int(np.argmin(np.abs(WL - w)))
sample = cube.rows(0, cube.nl, max(ST, cube.nl // 200))[:, ::max(1, cube.ns // 400), bi(800)]
v = sample[np.isfinite(sample) & (sample > 0)]
SCALE = (1e4 if np.percentile(v, 95) > 2 else 1.0) if REFLECTANCE_SCALE == 'auto' else float(REFLECTANCE_SCALE)
print(f'processing grid {NS} x {NL} (SPATIAL_STEP={ST}); reflectance divided by {SCALE:g}')

## 8 Pixel loop
The image is read `BLOCK_ROWS` rows at a time; for the crop pixels of each block the model computes its input, features and prediction. Values are NaN outside the crop.

In [ ]:
VAL = np.full((NL, NS), np.nan, np.float32); LAB = np.full((NL, NS), np.nan, np.float32); CROP = np.zeros((NL, NS), bool); t0 = time.time(); n_crop = 0
for r0 in range(0, cube.nl, BLOCK_ROWS * ST):
    r1 = min(cube.nl, r0 + BLOCK_ROWS * ST); blk = cube.rows(r0, r1, ST) / SCALE
    nd = (blk[..., bi(898)] - blk[..., bi(684)]) / (blk[..., bi(898)] + blk[..., bi(684)] + 1e-12)
    valid = np.isfinite(blk).all(2) & (blk[..., bi(800)] > 0) & (blk.mean(2) > 0) & (nd > NDVI_MIN)
    i0 = r0 // ST; nr = blk.shape[0]
    if valid.any():
        S = blk[valid]; CROP[i0:i0 + nr][valid] = True; n_crop += len(S)
        v, l = predict(S); VAL[i0:i0 + nr][valid] = v; LAB[i0:i0 + nr][valid] = l
    if (r0 // (BLOCK_ROWS * ST)) % 20 == 0 or r1 == cube.nl:
        el = time.time() - t0; print(f'rows {r1}/{cube.nl}  crop pixels {n_crop:,}  {el / 60:.1f} min  ETA {el / r1 * (cube.nl - r1) / 60:.1f} min')
base = os.path.join(OUT, KEY)
write_envi(base + '.dat', VAL, 'severity score (0-9)', 4, {'data ignore value': 'NaN'}); tif = write_tif(base + '.tif', VAL, np.nan)
print(f'done: {n_crop:,} crop pixels in {(time.time() - t0) / 60:.1f} min; GeoTIFF written: {bool(tif)}')

## 9 Map
Plot outlines and numbers are drawn from the ROI file. The title carries the model's accuracy on the 48 UAV plots so that an unreliable map is recognised at once.

In [ ]:
PLOTS = read_roi_xml(ROI_FILE) if ROI_FILE and os.path.exists(ROI_FILE) else {}
def outline(ax, numbers=True):
    for p, polys in PLOTS.items():
        for pg in polys: ax.plot((pg[:, 0] - 0.5) / ST, (pg[:, 1] - 0.5) / ST, color='k', lw=0.4)
        if numbers: c = (np.vstack(polys).mean(0) - 0.5) / ST; ax.text(c[0], c[1], str(p), fontsize=5, ha='center', va='center')
tag = f"UAV {MODEL['uav_metric'].replace('macro_F1', 'macro-F1').replace('R2', 'R²')} {MODEL['uav_value']:.2f}"
fig, axs = plt.subplots(1, 1, figsize=(8 * 1 + 1, 8 * NL / NS + 1), squeeze=False); axs = axs.ravel(); ax = axs[0]
im = ax.imshow(VAL, cmap='YlOrRd', vmin=0, vmax=9, interpolation='nearest'); lab = 'score (0-9)'
ttl = f"{MODEL['model']} severity score ({VARIANT} input; {tag}; NOT validated)"
ax.set_xlim(-0.5, NS - 0.5); ax.set_ylim(NL - 0.5, -0.5); outline(ax); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(ttl, fontsize=9); plt.colorbar(im, ax=ax, fraction=0.03, label=lab)
fig.tight_layout(); fp = os.path.join(OUT, KEY + '_map.png'); fig.savefig(fp, dpi=300); plt.close(fig); show(fp)

## 10 Per-plot accuracy
For each plot ROI: crop pixels inside the image and the plot value of the map, compared with the ASD leaf scores of 24 Feb. Score: mean predicted score vs the plot mean leaf score (R², RMSE, Pearson r). Plots 1 and 48 extend slightly beyond the image (95% and 89% of the plot inside); only their pixels inside the image are used.

In [ ]:
if PLOTS:
    yy, xx = np.mgrid[0:NL, 0:NS]; cx, cy = xx * ST + 0.5, yy * ST + 0.5; rows = []
    for p, polys in PLOTS.items():
        mk = np.zeros((NL, NS), bool)
        for pg in polys:
            sel = (cx >= pg[:, 0].min()) & (cx <= pg[:, 0].max()) & (cy >= pg[:, 1].min()) & (cy <= pg[:, 1].max())
            if sel.any(): mk[sel] |= MplPath(pg).contains_points(np.column_stack([cx[sel], cy[sel]]))
        mk &= CROP; v, l = VAL[mk], LAB[mk]; r = dict(plot=p, crop_pixels=int(mk.sum()))
        r.update(mean_score_pred=float(np.nanmean(v)) if len(v) else np.nan)
        rows.append(r)
    PS = pd.DataFrame(rows).merge(TRUTH, on='plot', how='left'); PS = PS[PS.crop_pixels > 0]
    e = PS.mean_score_pred - PS.mean_score
    S_ = dict(R2=1 - (e ** 2).sum() / ((PS.mean_score - PS.mean_score.mean()) ** 2).sum(), RMSE=float(np.sqrt((e ** 2).mean())), r_with_score=PS[['mean_score_pred', 'mean_score']].corr().iloc[0, 1])
    PS.to_csv(os.path.join(OUT, KEY + '_per_plot.csv'), index=False); pd.DataFrame([dict(model=KEY, plots=len(PS), **S_)]).to_csv(os.path.join(OUT, KEY + '_accuracy.csv'), index=False)
    print(PS.head(12).round(3).to_string(index=False)); print('...'); print(pd.Series(S_).round(3).to_string())
else:
    print('per-plot accuracy skipped (no ROI file)')

## 11 Outputs
In the results folder: the rasters (`C_XGBoost` (predicted severity score 0–9)), `C_XGBoost_map.png`, `C_XGBoost_per_plot.csv`, `C_XGBoost_accuracy.csv` and `run_info.json`. Open the `.dat/.hdr` or `.tif` in ENVI/QGIS – they carry the georeference of the image.

**References.** Apan et al. (2004); Barnes et al. (1989); Congalton (1991); Daughtry et al. (2000); Filella and Peñuelas (1994); Horler et al. (1983); Savitzky and Golay (1964); Sharma et al. (2019); Smith and Milton (1999); Westfall and Young (1993); Woo et al. (2023) – full entries in the transfer report.

In [ ]:
info = dict(model=KEY, image=DATA_FILE, spatial_step=ST, grid=[NS, NL], crop_pixels=int(n_crop), input=VARIANT, features=FEATURES, ndvi_min=NDVI_MIN, reflectance_scale=SCALE, minutes=round((time.time() - t0) / 60, 1))
json.dump(info, open(os.path.join(OUT, 'run_info.json'), 'w'), indent=1)
print('results in', OUT); print('\n'.join(sorted(os.listdir(OUT))))